 **This notebook imprement Ablation analysis of d-HyMoLAP (dimensionally consistent reformulation of HyMoLAP rainfall-runoff Model) run over CAMELS-GB dataset catchments with less than 10% of missing data.
**Author:** Lionel Cedric Gohouede

## 1. MOUNT GOOGLE DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
from numba import njit
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 3. FILTER DATA BY TIME PERIOD (1990-2014)

In [ ]:
data_dir = '/content/drive/MyDrive/Colab Notebooks/CAMELS_GB/'
start_date, end_date = '1990-01-01', '2014-12-31'

print("Loading, filtering, and aligning CSV files...\n")

# 1. Fast Load & Filter
df_pcp = pd.read_csv(f"{data_dir}pcp_mm.csv", index_col=0).loc[start_date:end_date]
df_pet = pd.read_csv(f"{data_dir}pet_mm.csv", index_col=0).loc[start_date:end_date]
df_q   = pd.read_csv(f"{data_dir}q_cms_obs.csv", index_col=0).loc[start_date:end_date]

# 2. Fast Alignment
common_stations = sorted(list(set(df_pcp.columns) & set(df_pet.columns) & set(df_q.columns)))
df_pcp = df_pcp[common_stations]
df_pet = df_pet[common_stations]
df_q   = df_q[common_stations]

print(f"✅ Data aligned. Total common stations: {len(common_stations)}")

# ============================================
# Memory-Optimized Wrapper Classes
# ============================================
class SimpleArray:
    __slots__ = ['array']  # Prevents massive RAM bloat
    def __init__(self, array):
        self.array = array
    def to_numpy(self):
        return self.array

class StationData:
    __slots__ = ['data']
    def __init__(self, data_dict):
        self.data = data_dict
    def sel(self, dynamic_features=None):
        return SimpleArray(self.data[dynamic_features])

# ============================================
# Ultra-Fast Dictionary Construction
# ============================================
print("Building ds_recent dictionary...\n")

# Convert DataFrames to 2D NumPy matrices once (orders of magnitude faster than column iteration)
pcp_arr = df_pcp.to_numpy()
pet_arr = df_pet.to_numpy()
q_arr   = df_q.to_numpy()
date_arr = pd.to_datetime(df_pcp.index).to_numpy()

# Build the dictionary using fast index slicing
ds_recent = {
    st: StationData({
        'pcp_mm': pcp_arr[:, i],
        'pet_mm': pet_arr[:, i],
        'q_cms_obs': q_arr[:, i],
        'date': date_arr
    })
    for i, st in enumerate(common_stations)
}

print(f"✅ Dictionary built successfully!")

# ============================================
# Validation Test
# ============================================
test_station = common_stations[0]
print(f"\nTesting data access for station: {test_station}")

Q_obs = ds_recent[test_station].sel(dynamic_features="q_cms_obs").to_numpy()
P     = ds_recent[test_station].sel(dynamic_features="pcp_mm").to_numpy()
PET   = ds_recent[test_station].sel(dynamic_features="pet_mm").to_numpy()

print(f"✅ Extraction works correctly!")
print(f"   Q_obs shape: {Q_obs.shape}")
print(f"   Q_obs - min: {np.nanmin(Q_obs):.2f}, max: {np.nanmax(Q_obs):.2f}, mean: {np.nanmean(Q_obs):.2f}")
print(f"   Q_obs Missing: {np.sum(np.isnan(Q_obs))} ({np.sum(np.isnan(Q_obs))/len(Q_obs)*100:.1f}%)")

Loading, filtering, and aligning CSV files...

✅ Data aligned. Total common stations: 671
Building ds_recent dictionary...

✅ Dictionary built successfully!

Testing data access for station: 10002
✅ Extraction works correctly!
   Q_obs shape: (9131,)
   Q_obs - min: 0.80, max: 94.41, mean: 5.18
   Q_obs Missing: 0 (0.0%)


## 4. MAIN CODE

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from numba import njit

@njit
def dHyMoLAP_Model(params, Q0, q):
    mu, lambda_, Qs, qs = params[0], params[1], params[2], params[3]
    N = len(q)
    k = np.zeros(N)
    x = np.zeros(N)

    if Qs <= 0.0 or qs <= 0.0 or lambda_ <= 0.0 or mu <= 0.0:
        return np.full(N, np.nan)

    k[0] = Q0 / Qs
    mu_lam = mu / lambda_
    one_m_mu_lam = 1.0 - mu_lam
    inv_lam = 1.0 / lambda_
    pow_term = 2.0 * mu - 1.0

    for t in range(N - 1):
        r_next = q[t+1] / qs

        if np.isnan(r_next):
            k[t+1] = k[t]
            x[t+1] = x[t]
            continue

        if r_next > 0.0:
            x[t+1] = x[t] + mu_lam * r_next
        else:
            x[t+1] = one_m_mu_lam * x[t]

        k_base = max(0.0, k[t])
        k[t+1] = max(0.0, k[t] - mu_lam * (k_base ** pow_term) + inv_lam * x[t+1] * r_next)

    return k * Qs

def NSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    valid_obs, valid_sim = obs[mask], sim[mask]
    if valid_obs.size == 0 or np.var(valid_obs) == 0.0:
        return np.nan
    return 1.0 - (np.sum((valid_sim - valid_obs)**2) / np.sum((valid_obs - np.mean(valid_obs))**2))

def compute_all_metrics(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    o = obs[mask]
    s = sim[mask]
    L = o.size

    if L == 0:
        return dict(NSE=np.nan, NNSE=np.nan, RMSE=np.nan, PBIAS=np.nan, FHV=np.nan, FLV=np.nan, KGE=np.nan)

    obs_mean = np.mean(o)
    sim_mean = np.mean(s)
    denom = np.sum((o - obs_mean) ** 2)

    nse = np.nan if denom == 0.0 else 1.0 - np.sum((s - o) ** 2) / denom
    nnse = 1.0 / (2.0 - nse) if np.isfinite(nse) else np.nan
    rmse = np.sqrt(np.mean((s - o) ** 2))

    sum_obs = np.sum(o)
    pbias = np.sum(s - o) / sum_obs if sum_obs != 0.0 else np.nan

    order_desc = np.argsort(o)[::-1]
    n_hv = max(1, int(round(0.02 * L)))
    idx_hv = order_desc[:n_hv]
    sum_obs_hv = np.sum(o[idx_hv])
    fhv = np.sum(s[idx_hv] - o[idx_hv]) / sum_obs_hv if sum_obs_hv != 0.0 else np.nan

    order_asc = np.argsort(o)
    n_lv = max(1, int(round(0.30 * L)))
    idx_lv = order_asc[:n_lv]
    sum_obs_lv = np.sum(o[idx_lv])
    flv = np.sum(s[idx_lv] - o[idx_lv]) / sum_obs_lv if sum_obs_lv != 0.0 else np.nan

    sigma_obs = np.std(o)
    sigma_sim = np.std(s)
    if sigma_obs == 0.0 or sigma_sim == 0.0 or obs_mean == 0.0:
        kge = np.nan
    else:
        r = np.corrcoef(o, s)[0, 1]
        alpha = sigma_sim / sigma_obs
        beta = sim_mean / obs_mean
        kge = np.nan if not np.isfinite(r) else 1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)

    return dict(NSE=nse, NNSE=nnse, RMSE=rmse, PBIAS=pbias, FHV=fhv, FLV=flv, KGE=kge)

def obj_base(p, Q0, q, obs):
    nse = NSE(obs, dHyMoLAP_Model(np.array([p[0], p[1], p[2], p[3]], dtype=np.float64), Q0, q))
    return 1.0 - nse if np.isfinite(nse) else 1e9

def obj_fix_qs(p, Q0, q, obs):
    nse = NSE(obs, dHyMoLAP_Model(np.array([p[0], p[1], p[2], 1.0], dtype=np.float64), Q0, q))
    return 1.0 - nse if np.isfinite(nse) else 1e9

def obj_fix_Qs(p, Q0, q, obs):
    nse = NSE(obs, dHyMoLAP_Model(np.array([p[0], p[1], 1.0, p[2]], dtype=np.float64), Q0, q))
    return 1.0 - nse if np.isfinite(nse) else 1e9

def obj_fix_both(p, Q0, q, obs):
    nse = NSE(obs, dHyMoLAP_Model(np.array([p[0], p[1], 1.0, 1.0], dtype=np.float64), Q0, q))
    return 1.0 - nse if np.isfinite(nse) else 1e9

all_stations = list(ds_recent.keys())
b1_ratio = 0.7
max_missing_ratio = 0.1

MU_BOUNDS = (0.5, 5.0)
LAMBDA_BOUNDS = (1e-3, 200.0)

results_base = {}
results_fix_qs = {}
results_fix_Qs = {}
results_fix_both = {}

for station_id in all_stations:
    Q_obs = ds_recent[station_id].sel(dynamic_features="q_cms_obs").to_numpy()
    P     = ds_recent[station_id].sel(dynamic_features="pcp_mm").to_numpy()
    PET   = ds_recent[station_id].sel(dynamic_features="pet_mm").to_numpy()

    q = np.maximum(0.0, P - PET)
    N = len(Q_obs)

    if N == 0 or np.all(np.isnan(Q_obs)) or (np.sum(np.isnan(Q_obs)) / N) > max_missing_ratio:
        continue

    b1 = int(N * b1_ratio)
    Q0 = Q_obs[0] if not np.isnan(Q_obs[0]) else np.nanmean(Q_obs[:10])
    if not np.isfinite(Q0):
        continue

    q_train = q[:b1]
    Q_obs_train = Q_obs[:b1]

    Qs_max = np.nanmax(Q_obs_train) if np.any(~np.isnan(Q_obs_train)) else np.nan
    qs_max = np.nanmax(q_train) if np.any(~np.isnan(q_train)) else np.nan

    if not np.isfinite(Qs_max) or not np.isfinite(qs_max):
        continue

    Qs_mean, qs_mean = np.clip(np.nanmean(Q_obs_train), 1e-3, 2.0*Qs_max), np.clip(np.nanmean(q_train), 1e-3, 2.0*qs_max)
    opt_opts = {'maxiter': 2500, 'disp': False}

    # 1. Base (Both qs and Qs optimized)
    res1 = minimize(obj_base, np.array([1.1, 20.0, Qs_mean, qs_mean]), args=(Q0, q_train, Q_obs_train),
                    method="Nelder-Mead", bounds=[MU_BOUNDS, LAMBDA_BOUNDS, (1e-3, 2.0*Qs_max), (1e-3, 2.0*qs_max)], options=opt_opts)
    p_base = np.array(res1.x, dtype=np.float64)

    # 2. Fix qs = 1
    res2 = minimize(obj_fix_qs, np.array([1.1, 20.0, Qs_mean]), args=(Q0, q_train, Q_obs_train),
                    method="Nelder-Mead", bounds=[MU_BOUNDS, LAMBDA_BOUNDS, (1e-3, 2.0*Qs_max)], options=opt_opts)
    p_fix_qs = np.array([res2.x[0], res2.x[1], res2.x[2], 1.0], dtype=np.float64)

    # 3. Fix Qs = 1
    res3 = minimize(obj_fix_Qs, np.array([1.1, 20.0, qs_mean]), args=(Q0, q_train, Q_obs_train),
                    method="Nelder-Mead", bounds=[MU_BOUNDS, LAMBDA_BOUNDS, (1e-3, 2.0*qs_max)], options=opt_opts)
    p_fix_Qs = np.array([res3.x[0], res3.x[1], 1.0, res3.x[2]], dtype=np.float64)

    # 4. Fix Both = 1
    res4 = minimize(obj_fix_both, np.array([1.1, 20.0]), args=(Q0, q_train, Q_obs_train),
                    method="Nelder-Mead", bounds=[MU_BOUNDS, LAMBDA_BOUNDS], options=opt_opts)
    p_fix_both = np.array([res4.x[0], res4.x[1], 1.0, 1.0], dtype=np.float64)

    configs = [
        (results_base, p_base),
        (results_fix_qs, p_fix_qs),
        (results_fix_Qs, p_fix_Qs),
        (results_fix_both, p_fix_both)
    ]

    for res_dict, params in configs:
        Qsim = dHyMoLAP_Model(params, Q0, q)
        m_tr = compute_all_metrics(Q_obs_train, Qsim[:b1])
        m_vl = compute_all_metrics(Q_obs[b1:], Qsim[b1:])

        res_dict[station_id] = {
            "NSE_train": m_tr["NSE"], "NSE_val": m_vl["NSE"],
            "NNSE_train": m_tr["NNSE"], "NNSE_val": m_vl["NNSE"],
            "RMSE_train": m_tr["RMSE"], "RMSE_val": m_vl["RMSE"],
            "PBIAS_train": m_tr["PBIAS"], "PBIAS_val": m_vl["PBIAS"],
            "FHV_train": m_tr["FHV"], "FHV_val": m_vl["FHV"],
            "FLV_train": m_tr["FLV"], "FLV_val": m_vl["FLV"],
            "KGE_train": m_tr["KGE"], "KGE_val": m_vl["KGE"]
        }

def print_statistics(results_dict, label):
    print(f"\n{'='*60}\n {label} \n{'='*60}")
    metrics = [
        ("NSE_train", "NSE Training"), ("NSE_val", "NSE Validation"),
        ("KGE_train", "KGE Training"), ("KGE_val", "KGE Validation"),
        ("RMSE_train", "RMSE Training"), ("RMSE_val", "RMSE Validation"),
        ("PBIAS_train", "PBIAS Training"), ("PBIAS_val", "PBIAS Validation"),
        ("FHV_train", "FHV Training"), ("FHV_val", "FHV Validation"),
        ("FLV_train", "FLV Training"), ("FLV_val", "FLV Validation")
    ]

    if not results_dict:
        print("No stations successfully processed.")
        return

    for key, name in metrics:
        vals = np.array([r[key] for r in results_dict.values()], dtype=float)
        valid_vals = vals[~np.isnan(vals)]

        if len(valid_vals) > 0:
            print(f"**{name:17s}** | Mean: {np.mean(valid_vals):>6.3f} | Median: {np.median(valid_vals):>6.3f} | "
                  f"Min: {np.min(valid_vals):>6.3f} | Max: {np.max(valid_vals):>6.3f} | "
                  f"5th: {np.percentile(valid_vals, 5):>6.3f} | 95th: {np.percentile(valid_vals, 95):>6.3f}")
        else:
            print(f"**{name:17s}** | All values are NaN")

print_statistics(results_base, "1. RUN CALIBRATION (Optimize Both qs and Qs)")
print_statistics(results_fix_qs, "2. ABLATION RUN (Fix qs = 1)")
print_statistics(results_fix_Qs, "3. ABLATION RUN (Fix Qs = 1)")
print_statistics(results_fix_both, "4. ABLATION RUN (Fix Both qs = 1 and Qs = 1)")


 1. RUN CALIBRATION (Optimize Both qs and Qs) 
**NSE Training     ** | Mean:  0.703 | Median:  0.718 | Min:  0.033 | Max:  0.922 | 5th:  0.531 | 95th:  0.827
**NSE Validation   ** | Mean:  0.650 | Median:  0.679 | Min: -1.764 | Max:  0.906 | 5th:  0.435 | 95th:  0.807
**KGE Training     ** | Mean:  0.764 | Median:  0.783 | Min: -0.060 | Max:  0.938 | 5th:  0.602 | 95th:  0.868
**KGE Validation   ** | Mean:  0.716 | Median:  0.741 | Min: -0.547 | Max:  0.947 | 5th:  0.488 | 95th:  0.863
**RMSE Training    ** | Mean:  4.901 | Median:  1.758 | Min:  0.017 | Max: 74.799 | 5th:  0.141 | 95th: 22.408
**RMSE Validation  ** | Mean:  5.315 | Median:  2.038 | Min:  0.022 | Max: 82.058 | 5th:  0.155 | 95th: 23.518
**PBIAS Training   ** | Mean:  0.004 | Median: -0.002 | Min: -0.273 | Max:  0.377 | 5th: -0.075 | 95th:  0.101
**PBIAS Validation ** | Mean:  0.027 | Median:  0.019 | Min: -0.358 | Max:  0.861 | 5th: -0.114 | 95th:  0.197
**FHV Training     ** | Mean: -0.324 | Median: -0.323 | Min: -0.